## Phase 3: Prediction Model Building & Rolling Retraining Pipeline
### Step 1 — Lag Target Pair Engineering & Evaluation Matrix Initialization

Creates feature-target pairs for $y_{t+1}$ forecasting and allocates
memory for daily prediction values across 3 retraining scenarios:
- Scenario A: WASSERSTEIN_60 (quarterly-batch control)
- Scenario B: WASSERSTEIN_120 (semester-batch control)
- Scenario C: ADWIN (pure-stream control)

**Anti-Look-Ahead Discipline:**
Target is created via `shift(-1)` so row $t$ pairs with $y_{t+1}$.
Last row is dropped (no future target available).
Warm-up: rows 0-240. Simulation runs from row 241 onward.

#### 1. Configuration & Imports

Reuses feature list from Phase 1. Adds Phase 3 specific parameters:
- `warmup_rows`: 240 (rows 0-240 = 241 rows of initial training)
- `xgb_window`: 250 (XGBoost rolling window on retrain)
- `oselm_hidden`: 100 (OS-ELM hidden layer size)
- `scenario_controls`: maps scenario → global drift signal name

In [ ]:
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd


@dataclass
class Config:
    data_path: str = "../dataset/processed/jkse_preprocessed.csv"
    output_dir: str = "../dataset/processed"
    features: list = field(default_factory=lambda: [
        "Log_Return", "Vol_20d", "Vol_60d", "EMA_5",
        "BB_Middle", "BB_Upper", "BB_Lower",
        "Momentum_5d", "Momentum_20d",
    ])
    batch_algorithms: list = field(default_factory=lambda: [
        "mysd", "ks", "psi", "wasserstein",
    ])
    stream_algorithms: list = field(default_factory=lambda: [
        "adwin", "kswin", "ph",
    ])
    batch_windows: list = field(default_factory=lambda: [60, 120])
    consensus_threshold: float = 1 / 3
    # Phase 3 specific
    warmup_rows: int = 240
    xgb_window: int = 250
    oselm_hidden: int = 100
    simulation_start: int = 241
    scenario_controls: dict = field(default_factory=lambda: {
        "A": "Global_wasserstein_60",
        "B": "Global_wasserstein_120",
        "C": "Global_adwin",
    })

In [ ]:
cfg = Config()

#### 2. Data Loading & Drift Signal Reconstruction

Loads the clean Phase 1 dataset. Rebuilds global drift decision vectors
(from Phase 2 scoreboards) for the 3 scenario controls.
The consensus rule: >= 1/3 of 9 features must fire simultaneously.

In [ ]:
df = pd.read_csv(cfg.data_path, index_col=0, parse_dates=True)
df = df.sort_index()

print(f"Loaded {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Date range: {df.index.min().date()}  to  {df.index.max().date()}")

##### 2a. Utility: Load global drift signal from Phase 2 scoreboard

Each scoreboard has 9 feature columns. Row is `1.0` (global drift)
if mean fraction of triggered features >= `consensus_threshold`.

In [ ]:
def load_global_signal(
    key: str,
    output_dir: str,
    features: list[str],
    threshold: float,
) -> pd.Series:
    path = Path(output_dir) / f"scoreboard_{key}.csv"
    sb = pd.read_csv(path, index_col=0, parse_dates=True)
    consensus = sb[features].mean(axis=1)
    is_drift = (consensus >= threshold).astype(float)
    is_drift[sb[features].isnull().all(axis=1)] = np.nan
    return is_drift

In [ ]:
drift_signals: dict[str, pd.Series] = {}
for scenario, signal_name in cfg.scenario_controls.items():
    # Extract algorithm key from signal_name (e.g. 'Global_wasserstein_60' -> 'wasserstein_60')
    key = signal_name.replace("Global_", "", 1)
    s = load_global_signal(key, cfg.output_dir, cfg.features, cfg.consensus_threshold)
    drift_signals[scenario] = s
    n_triggers = int(s.dropna().sum())
    print(f"Scenario {scenario} ({signal_name}): {n_triggers} global drift days")

#### 3. Target Engineering & X/y Split

- `Target_Log_Return = Log_Return.shift(-1)` pairs today's features with tomorrow's return
- Drop the last row (NaN target after shift)
- Split into matrix X (9 features) and vector y (Target_Log_Return)

**Edge case guard:** Verify no NaN leakage in simulation zone (row 241+).

In [ ]:
df["Target_Log_Return"] = df["Log_Return"].shift(-1)

n_before = len(df)
df.dropna(subset=["Target_Log_Return"], inplace=True)
n_dropped = n_before - len(df)

X = df[cfg.features].copy()
y = df["Target_Log_Return"].copy()

print(f"Rows before shift:       {n_before}")
print(f"Rows dropped (last row): {n_dropped}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print()

# Sanity: no NaN in simulation zone
assert X.iloc[cfg.simulation_start:].isnull().sum().sum() == 0, "NaN in X simulation zone"
assert y.iloc[cfg.simulation_start:].isnull().sum() == 0, "NaN in y simulation zone"
print("No NaN in simulation zone (row 241+): OK")

#### 4. Prediction Storage DataFrame

Allocates 7 columns on a DatetimeIndex:
- `y_true`: actual target (for evaluation in Phase 4)
- `Pred_XGB_A`, `Pred_OSELM_A`: Scenario A (WASSERSTEIN_60)
- `Pred_XGB_B`, `Pred_OSELM_B`: Scenario B (WASSERSTEIN_120)
- `Pred_XGB_C`, `Pred_OSELM_C`: Scenario C (ADWIN)

All prediction columns are NaN for rows 0-240 (warm-up),
initialized to `0.0` placeholder from row 241 onward
(will be overwritten by actual predictions during the prequential loop).

In [ ]:
pred_columns = []
for scenario in ["A", "B", "C"]:
    pred_columns.append(f"Pred_XGB_{scenario}")
    pred_columns.append(f"Pred_OSELM_{scenario}")

pred_df = pd.DataFrame(index=X.index, columns=["y_true"] + pred_columns, dtype=float)
pred_df["y_true"] = y

# Fill prediction columns with 0.0 from simulation_start onward
pred_df.iloc[: cfg.simulation_start, 1:] = np.nan
pred_df.iloc[cfg.simulation_start:, 1:] = 0.0

print(f"pred_df shape: {pred_df.shape}")
print(f"Columns: {list(pred_df.columns)}")
print(f"Warm-up rows (NaN): {int(pred_df.iloc[:cfg.simulation_start, 1:].isnull().all(axis=1).sum())}")
print(f"Simulation rows (0.0 placeholder): {len(pred_df) - cfg.simulation_start}")

#### 5. Validation Report

**Required outputs:**
1. Final dimensions of X and y after target-lag trimming
2. Alignment head check: print X and y at row 241 to confirm
   day-t features pair with day-(t+1) target (no look-ahead bias)
3. Confirm drift signals and pred_df are row-aligned

In [ ]:
print("=" * 60)
print("PHASE 3 STEP 1 — VALIDATION REPORT")
print("=" * 60)
print()

print("1. Final Matrix Dimensions")
print(f"   X shape: {X.shape}")
print(f"   y shape: {y.shape}")
print()

print("2. Alignment Head Check (row 241 = simulation start)")
print(f"   Simulation start date: {X.index[cfg.simulation_start].date()}")
print()
print("   X.iloc[241:244] (features at day t):")
print(X.iloc[cfg.simulation_start : cfg.simulation_start + 3].round(6).to_string())
print()
print("   y.iloc[241:244] (target = Log_Return at t+1):")
y_slice = y.iloc[cfg.simulation_start : cfg.simulation_start + 3]
print(y_slice.to_frame("Target_Log_Return").round(6).to_string())
print()

# Cross-check: y at row t should equal Log_Return at row t+1
t = cfg.simulation_start
match = np.isclose(y.iloc[t], X.iloc[t + 1]["Log_Return"])
print(f"   Cross-check: y[{t}] ({y.iloc[t]:.6f}) == Log_Return[{t+1}] ({X.iloc[t+1]['Log_Return']:.6f})? {'MATCH' if match else 'MISMATCH!'}")
print()

print("3. Drift Signal & pred_df Alignment")
for sc in ["A", "B", "C"]:
    s = drift_signals[sc]
    # Align to trimmed index
    s_aligned = s.reindex(X.index)
    n_sim = int(s_aligned.iloc[cfg.simulation_start:].sum())
    print(f"   Scenario {sc}: {n_sim} drift triggers in simulation zone")
print()
print(f"   pred_df rows: {len(pred_df)}, X rows: {len(X)} — match: {len(pred_df) == len(X)}")

#### 6. OS-ELM Class Definition

Online Sequential Extreme Learning Machine.

**Architecture:**
- Single hidden layer with `n_hidden` neurons, sigmoid activation
- Input weights $W_{in}$ randomly initialized from $U(-1, 1)$, never updated
- Bias $b$ randomly initialized from $U(0, 1)$, never updated
- Output weights $\beta$ solved analytically

**Training:**
- `fit(X_init, y_init)`: Batch initialization via pseudo-inverse
- `partial_fit(X_new, y_new)`: Recursive least squares (RLS) update
  using the Sherman-Morrison-Woodbury formula for $O(H^2)$ per step

**Prediction:**
- `predict(X)`: $H = \sigma(X \cdot W_{in} + b)$, output = $H \cdot \beta$

In [ ]:
class OSELM:
    """Online Sequential Extreme Learning Machine.

    Parameters
    ----------
    n_inputs : int
        Number of input features.
    n_hidden : int
        Number of hidden layer neurons.
    activation : str
        Activation function ('sigmoid' or 'relu').
    random_state : int or None
        Seed for reproducible random weights.
    """

    def __init__(
        self,
        n_inputs: int = 9,
        n_hidden: int = 100,
        activation: str = "sigmoid",
        random_state: int | None = 42,
    ):
        self.n_inputs = n_inputs
        self.n_hidden = n_hidden
        self.activation = activation
        self.random_state = random_state

        rng = np.random.default_rng(random_state)
        self.W_in = rng.uniform(-1.0, 1.0, size=(n_inputs, n_hidden))
        self.bias = rng.uniform(0.0, 1.0, size=(1, n_hidden))

        self.beta: np.ndarray | None = None
        self.P: np.ndarray | None = None  # RLS covariance matrix
        self._fitted = False

    def _activate(self, H: np.ndarray) -> np.ndarray:
        if self.activation == "sigmoid":
            H = np.clip(H, -500, 500)
            return 1.0 / (1.0 + np.exp(-H))
        if self.activation == "relu":
            return np.maximum(0.0, H)
        raise ValueError(f"Unknown activation: {self.activation}")

    def _hidden(self, X: np.ndarray) -> np.ndarray:
        """Compute hidden layer output matrix H (N x n_hidden)."""
        return self._activate(X @ self.W_in + self.bias)

    def fit(self, X: np.ndarray, y: np.ndarray):
        """Initial batch training via pseudo-inverse."""
        H = self._hidden(X)
        H_pinv = np.linalg.pinv(H)
        self.beta = H_pinv @ y
        # Initialize RLS covariance: P = (H^T H)^{-1}
        HtH = H.T @ H
        self.P = np.linalg.inv(HtH + 1e-8 * np.eye(self.n_hidden))
        self._fitted = True
        return self

    def partial_fit(self, X_new: np.ndarray, y_new: np.ndarray):
        """Online update using recursive least squares.

        Implements the Sherman-Morrison-Woodbury formula for
        efficient O(H^2) per-sample update.
        """
        if not self._fitted:
            return self.fit(X_new, y_new)

        H = self._hidden(X_new)  # (N x H)
        for i in range(H.shape[0]):
            h = H[i : i + 1, :]  # (1 x H)
            y_i = y_new[i : i + 1]

            # Gain vector: K = P @ h^T / (1 + h @ P @ h^T)
            Ph = self.P @ h.T  # (H x 1)
            denom = 1.0 + (h @ Ph).item()
            K = Ph / denom

            # Update beta
            error = y_i - (h @ self.beta).item()
            self.beta = self.beta + (K.flatten() * error)

            # Update covariance: P = (I - K @ h) @ P
            self.P = self.P - K @ (h @ self.P)

        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        if not self._fitted:
            raise RuntimeError("OSELM not fitted — call .fit() first")
        H = self._hidden(X)
        return H @ self.beta

##### 6a. OS-ELM Smoke Test

Quick sanity check: train on warm-up data, verify shapes converge.

In [ ]:
oselm = OSELM(n_inputs=9, n_hidden=cfg.oselm_hidden, activation="sigmoid")

X_warm = X.iloc[: cfg.simulation_start].values
y_warm = y.iloc[: cfg.simulation_start].values

oselm.fit(X_warm, y_warm)
y_pred = oselm.predict(X_warm[:5])

print(f"OS-ELM smoke test passed:")
print(f"  Input shape:  {X_warm.shape}")
print(f"  Beta shape:   {oselm.beta.shape}")
print(f"  P shape:      {oselm.P.shape}")
print(f"  Predict shape: {y_pred.shape}")
print(f"  Sample preds: {y_pred[:3].round(6)}")

# Partial fit test
oselm.partial_fit(X.iloc[241:245].values, y.iloc[241:245].values)
y_pred2 = oselm.predict(X.iloc[241:245].values)
print(f"  After partial_fit (4 rows): {y_pred2[:2].round(6)}")

#### 7. Persist Phase 3 Step 1 Artifacts

Saves `pred_df`, drift signals, and X/y arrays as CSV for future steps.

In [ ]:
# Save prediction DataFrame
pred_path = Path(cfg.output_dir) / "predictions_step1.csv"
pred_df.to_csv(pred_path)
print(f"Saved pred_df: {pred_path} ({pred_path.stat().st_size / 1024:.1f} KB)")

# Save X and y
X.to_csv(Path(cfg.output_dir) / "X_features.csv")
y.to_frame("Target_Log_Return").to_csv(Path(cfg.output_dir) / "y_target.csv")
print(f"Saved X ({X.shape}) and y ({y.shape})")

# Save drift signals (aligned to trimmed index)
drift_aligned = pd.DataFrame(index=X.index)
for sc, s in drift_signals.items():
    drift_aligned[f"drift_{sc}"] = s.reindex(X.index)
drift_aligned.to_csv(Path(cfg.output_dir) / "drift_signals_aligned.csv")
print(f"Saved drift signals ({drift_aligned.shape})")

## Phase 3: Step 2 — Prequential Simulation Scenario A (WASSERSTEIN_60)

Daily test-accumulate-retrain loop from row 241 to end.
Controlled by the `Global_wasserstein_60` drift signal.

**Prequential Order (per day t):**
1. **Test** — predict day t's return with current models
2. **Accumulate** — enqueue (X[t], y[t]) into OS-ELM buffer
3. **Sensor + Retrain** — if drift flagged at t:
   - XGBoost: cold-restart on rolling 250-day window
   - OS-ELM: partial_fit on buffer, then clear buffer

In [ ]:
from xgboost import XGBRegressor
import time

# Reload data from Step 1 CSVs
X = pd.read_csv(Path(cfg.output_dir) / "X_features.csv", index_col=0)
y = pd.read_csv(Path(cfg.output_dir) / "y_target.csv", index_col=0).squeeze("columns")
pred_df = pd.read_csv(Path(cfg.output_dir) / "predictions_step1.csv", index_col=0)
drift_signals = pd.read_csv(Path(cfg.output_dir) / "drift_signals_aligned.csv", index_col=0)
drift_A = drift_signals["drift_A"]

print(f"X: {X.shape}, y: {y.shape}")
print(f"pred_df: {pred_df.shape}")
print(f"drift_A triggers in simulation zone: {int(drift_A.iloc[cfg.simulation_start:].sum())}")

# Warm-up fit
xgb = XGBRegressor(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, random_state=42,
)
xgb.fit(X.iloc[:cfg.simulation_start], y.iloc[:cfg.simulation_start])

oselm = OSELM(n_inputs=9, n_hidden=cfg.oselm_hidden, activation="sigmoid")
oselm.fit(X.iloc[:cfg.simulation_start].values, y.iloc[:cfg.simulation_start].values)

# OS-ELM streaming buffers
buf_X: list[np.ndarray] = []
buf_y: list[float] = []

# Timer
start_time = time.perf_counter()
retrain_count = 0

print("Ready. Starting prequential loop...")

In [ ]:
for t in range(cfg.simulation_start, len(X)):
    x_t = X.iloc[t : t + 1]
    y_t = y.iloc[t]

    # Phase 1: Test
    p_xgb = float(xgb.predict(x_t)[0])
    p_osl = float(oselm.predict(x_t.values)[0])
    pred_df.iloc[t, 1] = p_xgb    # Pred_XGB_A
    pred_df.iloc[t, 2] = p_osl    # Pred_OSELM_A

    # Phase 2: Accumulate into OS-ELM buffer
    buf_X.append(x_t.values[0])
    buf_y.append(y_t)

    # Phase 3: Sensor + Retrain
    if drift_A.iloc[t] == 1.0:
        # XGBoost cold restart on rolling window
        lo = max(0, t + 1 - cfg.xgb_window)
        xgb = XGBRegressor(
            n_estimators=100, max_depth=3, learning_rate=0.05,
            subsample=0.8, random_state=42,
        )
        xgb.fit(X.iloc[lo : t + 1], y.iloc[lo : t + 1])

        # OS-ELM incremental update on accumulated buffer
        oselm.partial_fit(np.array(buf_X), np.array(buf_y))
        buf_X.clear()
        buf_y.clear()
        retrain_count += 1

elapsed = time.perf_counter() - start_time
print(f"Total compute time: {elapsed:.2f}s")

In [ ]:
print("=" * 60)
print("PHASE 3 STEP 2 — SCENARIO A REPORT")
print("=" * 60)
print()
print(f"Total compute time: {elapsed:.2f}s")
print(f"Retraining count (XGBoost + OS-ELM): {retrain_count}")
expected = int(drift_A.iloc[cfg.simulation_start:].sum())
print(f"Expected (drift_A triggers): {expected}")
print(f"Match: {retrain_count == expected}")
print()
print("Tail of pred_df (Scenario A columns) — row by row:")
tail_df = pred_df[["y_true", "Pred_XGB_A", "Pred_OSELM_A"]].tail()
for idx, row in tail_df.iterrows():
    print(f"  {idx}  {row['y_true']:+.6f}  {row['Pred_XGB_A']:+.6f}  {row['Pred_OSELM_A']:+.6f}")
print()
# Health check: predictions must not be constant
sim = pred_df.iloc[cfg.simulation_start:]
xgb_std = sim["Pred_XGB_A"].std()
osl_std = sim["Pred_OSELM_A"].std()
print()
print(f"Pred_XGB_A std (simulation zone): {xgb_std:.6f}")
print(f"Pred_OSELM_A std (simulation zone): {osl_std:.6f}")
warn = xgb_std < 1e-8 or osl_std < 1e-8
print(f"Degeneration risk: {'NONE' if not warn else 'WARNING — constant predictions'}")
print()
# Persist
out = Path(cfg.output_dir) / "predictions_step2.csv"
pred_df.to_csv(out)
tail_df.to_csv(Path(cfg.output_dir) / "pred_tail_step2.csv")
print(f"Saved pred_df: {out}")


## Phase 3: Step 3 — Prequential Simulation Scenario B (WASSERSTEIN_120)

Daily test-accumulate-retrain loop from row 241 to end.
Controlled by the `Global_wasserstein_120` drift signal.

**Prequential Order (per day t):**
1. **Test** — predict day t's return with current models
2. **Accumulate** — enqueue (X[t], y[t]) into OS-ELM buffer
3. **Sensor + Retrain** — if drift flagged at t:
   - XGBoost: cold-restart on rolling 250-day window
   - OS-ELM: partial_fit on buffer, then clear buffer

In [ ]:
from xgboost import XGBRegressor
import time

X = pd.read_csv(Path(cfg.output_dir) / "X_features.csv", index_col=0)
y = pd.read_csv(Path(cfg.output_dir) / "y_target.csv", index_col=0).squeeze("columns")
pred_df = pd.read_csv(Path(cfg.output_dir) / "predictions_step2.csv", index_col=0)
drift_signals = pd.read_csv(Path(cfg.output_dir) / "drift_signals_aligned.csv", index_col=0)
drift_B = drift_signals["drift_B"]

print(f"X: {X.shape}, y: {y.shape}")
print(f"pred_df: {pred_df.shape}")
print(f"drift_B triggers in simulation zone: {int(drift_B.iloc[cfg.simulation_start:].sum())}")

# Fresh models — no memory from Step 2
xgb = XGBRegressor(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, random_state=42,
)
xgb.fit(X.iloc[:cfg.simulation_start], y.iloc[:cfg.simulation_start])

oselm = OSELM(n_inputs=9, n_hidden=cfg.oselm_hidden, activation="sigmoid")
oselm.fit(X.iloc[:cfg.simulation_start].values, y.iloc[:cfg.simulation_start].values)

buf_X, buf_y = [], []
start_time = time.perf_counter()
retrain_count = 0

print("Ready. Starting prequential loop...")

In [ ]:
for t in range(cfg.simulation_start, len(X)):
    x_t = X.iloc[t : t + 1]
    y_t = y.iloc[t]

    # Phase 1: Test
    p_xgb = float(xgb.predict(x_t)[0])
    p_osl = float(oselm.predict(x_t.values)[0])
    pred_df.iloc[t, 3] = p_xgb    # Pred_XGB_B
    pred_df.iloc[t, 4] = p_osl    # Pred_OSELM_B

    # Phase 2: Accumulate into OS-ELM buffer
    buf_X.append(x_t.values[0])
    buf_y.append(y_t)

    # Phase 3: Sensor + Retrain
    if drift_B.iloc[t] == 1.0:
        lo = max(0, t + 1 - cfg.xgb_window)
        xgb = XGBRegressor(
            n_estimators=100, max_depth=3, learning_rate=0.05,
            subsample=0.8, random_state=42,
        )
        xgb.fit(X.iloc[lo : t + 1], y.iloc[lo : t + 1])

        # OS-ELM incremental update on accumulated buffer
        oselm.partial_fit(np.array(buf_X), np.array(buf_y))
        buf_X.clear()
        buf_y.clear()
        retrain_count += 1

elapsed = time.perf_counter() - start_time
print(f"Total compute time: {elapsed:.2f}s")

In [ ]:
print("=" * 60)
print("PHASE 3 STEP 3 — SCENARIO B REPORT")
print("=" * 60)
print()
print(f"Total compute time: {elapsed:.2f}s")
print(f"Retraining count (XGBoost + OS-ELM): {retrain_count}")
expected = int(drift_B.iloc[cfg.simulation_start:].sum())
print(f"Expected (drift_B triggers): {expected}")
print(f"Match: {retrain_count == expected}")
print()
print("Tail of pred_df (Scenario B columns) — row by row:")
tail_df = pred_df[["y_true", "Pred_XGB_B", "Pred_OSELM_B"]].tail()
for idx, row in tail_df.iterrows():
    print(f"  {idx}  {row['y_true']:+.6f}  {row['Pred_XGB_B']:+.6f}  {row['Pred_OSELM_B']:+.6f}")
print()
# Health check: predictions must not be constant
sim = pred_df.iloc[cfg.simulation_start:]
xgb_std = sim["Pred_XGB_B"].std()
osl_std = sim["Pred_OSELM_B"].std()
print()
print(f"Pred_XGB_B std (simulation zone): {xgb_std:.6f}")
print(f"Pred_OSELM_B std (simulation zone): {osl_std:.6f}")
warn = xgb_std < 1e-8 or osl_std < 1e-8
print(f"Degeneration risk: {'NONE' if not warn else 'WARNING — constant predictions'}")
print()
# Persist
out = Path(cfg.output_dir) / "predictions_step3.csv"
pred_df.to_csv(out)
tail_df.to_csv(Path(cfg.output_dir) / "pred_tail_step3.csv")
print(f"Saved pred_df: {out}")

## Phase 3: Step 4 — Prequential Simulation Scenario C (ADWIN)

Daily test-accumulate-retrain loop from row 241 to end.
Controlled by the ADWIN (river) pure stream drift signal.

**Prequential Order (per day t):**
1. **Test** — predict day t's return with current models
2. **Accumulate** — enqueue (X[t], y[t]) into OS-ELM buffer
3. **Sensor + Retrain** — if drift flagged at t:
   - XGBoost: cold-restart on rolling 250-day window
   - OS-ELM: partial_fit on buffer, then clear buffer

In [ ]:
from xgboost import XGBRegressor
import time

X = pd.read_csv(Path(cfg.output_dir) / "X_features.csv", index_col=0)
y = pd.read_csv(Path(cfg.output_dir) / "y_target.csv", index_col=0).squeeze("columns")
pred_df = pd.read_csv(Path(cfg.output_dir) / "predictions_step3.csv", index_col=0)
drift_signals = pd.read_csv(Path(cfg.output_dir) / "drift_signals_aligned.csv", index_col=0)
drift_C = drift_signals["drift_C"]

print(f"X: {X.shape}, y: {y.shape}")
print(f"pred_df: {pred_df.shape}")
print(f"drift_C triggers in simulation zone: {int(drift_C.iloc[cfg.simulation_start:].sum())}")

# Fresh models — no memory from prior steps
xgb = XGBRegressor(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, random_state=42,
)
xgb.fit(X.iloc[:cfg.simulation_start], y.iloc[:cfg.simulation_start])

oselm = OSELM(n_inputs=9, n_hidden=cfg.oselm_hidden, activation="sigmoid")
oselm.fit(X.iloc[:cfg.simulation_start].values, y.iloc[:cfg.simulation_start].values)

buf_X, buf_y = [], []
start_time = time.perf_counter()
retrain_count = 0

print("Ready. Starting prequential loop...")

In [ ]:
for t in range(cfg.simulation_start, len(X)):
    x_t = X.iloc[t : t + 1]
    y_t = y.iloc[t]

    # Phase 1: Test
    p_xgb = float(xgb.predict(x_t)[0])
    p_osl = float(oselm.predict(x_t.values)[0])
    pred_df.iloc[t, 5] = p_xgb    # Pred_XGB_C
    pred_df.iloc[t, 6] = p_osl    # Pred_OSELM_C

    # Phase 2: Accumulate into OS-ELM buffer
    buf_X.append(x_t.values[0])
    buf_y.append(y_t)

    # Phase 3: Sensor + Retrain
    if drift_C.iloc[t] == 1.0:
        lo = max(0, t + 1 - cfg.xgb_window)
        xgb = XGBRegressor(
            n_estimators=100, max_depth=3, learning_rate=0.05,
            subsample=0.8, random_state=42,
        )
        xgb.fit(X.iloc[lo : t + 1], y.iloc[lo : t + 1])

        # OS-ELM incremental update on accumulated buffer
        oselm.partial_fit(np.array(buf_X), np.array(buf_y))
        buf_X.clear()
        buf_y.clear()
        retrain_count += 1

elapsed = time.perf_counter() - start_time
print(f"Total compute time: {elapsed:.2f}s")

In [ ]:
print("=" * 60)
print("PHASE 3 STEP 4 — SCENARIO C REPORT (ADWIN)")
print("=" * 60)
print()
print(f"Total compute time: {elapsed:.2f}s")
print(f"Retraining count (XGBoost + OS-ELM): {retrain_count}")
expected = int(drift_C.iloc[cfg.simulation_start:].sum())
print(f"Expected (drift_C triggers): {expected}")
print(f"Match: {retrain_count == expected}")
print()
print("Tail of pred_df (Scenario C columns) — row by row:")
tail_df = pred_df[["y_true", "Pred_XGB_C", "Pred_OSELM_C"]].tail()
for idx, row in tail_df.iterrows():
    print(f"  {idx}  {row['y_true']:+.6f}  {row['Pred_XGB_C']:+.6f}  {row['Pred_OSELM_C']:+.6f}")
print()
# Health check: predictions must not be constant
sim = pred_df.iloc[cfg.simulation_start:]
xgb_std = sim["Pred_XGB_C"].std()
osl_std = sim["Pred_OSELM_C"].std()
print()
print(f"Pred_XGB_C std (simulation zone): {xgb_std:.6f}")
print(f"Pred_OSELM_C std (simulation zone): {osl_std:.6f}")
warn = xgb_std < 1e-8 or osl_std < 1e-8
print(f"Degeneration risk: {'NONE' if not warn else 'WARNING — constant predictions'}")
print()
# Persist
out = Path(cfg.output_dir) / "predictions_step4.csv"
pred_df.to_csv(out)
tail_df.to_csv(Path(cfg.output_dir) / "pred_tail_step4.csv")
print(f"Saved pred_df: {out}")

## Phase 3: Step 5 — Static Model Baseline (Upper Error Bound)

Fit XGBoost and OS-ELM **once** on warm-up (rows 0-240), then predict
all 3688 simulation rows without any retraining call.

This is the "no-update" baseline — the upper error bound.
If drift-driven retraining can't outperform this, retraining is pointless.

In [ ]:
from xgboost import XGBRegressor
import time

X = pd.read_csv(Path(cfg.output_dir) / "X_features.csv", index_col=0)
y = pd.read_csv(Path(cfg.output_dir) / "y_target.csv", index_col=0).squeeze("columns")
pred_df = pd.read_csv(Path(cfg.output_dir) / "predictions_step4.csv", index_col=0)

pred_df["Pred_XGB_Static"] = 0.0
pred_df["Pred_OSELM_Static"] = 0.0

xgb = XGBRegressor(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, random_state=42,
)
xgb.fit(X.iloc[:cfg.simulation_start], y.iloc[:cfg.simulation_start])

oselm = OSELM(n_inputs=9, n_hidden=cfg.oselm_hidden, activation="sigmoid")
oselm.fit(X.iloc[:cfg.simulation_start].values, y.iloc[:cfg.simulation_start].values)

print(f"X: {X.shape}, y: {y.shape}, pred_df: {pred_df.shape}")
print("Models fitted once on warm-up. Starting static prediction loop...")

In [ ]:
start_time = time.perf_counter()
retrain_count = 0

for t in range(cfg.simulation_start, len(X)):
    x_t = X.iloc[t : t + 1]
    pred_df.iloc[t, 7] = float(xgb.predict(x_t)[0])
    pred_df.iloc[t, 8] = float(oselm.predict(x_t.values)[0])

elapsed = time.perf_counter() - start_time
print(f"Total compute time: {elapsed:.2f}s")

In [ ]:
print("=" * 60)
print("PHASE 3 STEP 5 — STATIC MODEL (UPPER ERROR BOUND)")
print("=" * 60)
print()
print(f"Total compute time: {elapsed:.2f}s")
print("Retraining count: 0 (frozen model, no retraining)")
print()

sim = pred_df.iloc[cfg.simulation_start:]
xgb_std = sim["Pred_XGB_Static"].std()
osl_std = sim["Pred_OSELM_Static"].std()
print(f"Pred_XGB_Static std (simulation zone): {xgb_std:.6f}")
print(f"Pred_OSELM_Static std (simulation zone): {osl_std:.6f}")
warn = xgb_std < 1e-8 or osl_std < 1e-8
print(f"Degeneration risk: {'NONE' if not warn else 'WARNING — constant predictions'}")
print()

print("Tail of pred_df (Static columns) — row by row:")
tail_df = pred_df[["y_true", "Pred_XGB_Static", "Pred_OSELM_Static"]].tail()
for idx, row in tail_df.iterrows():
    print(f"  {idx}  {row['y_true']:+.6f}  {row['Pred_XGB_Static']:+.6f}  {row['Pred_OSELM_Static']:+.6f}")
print()

out = Path(cfg.output_dir) / "predictions_step5.csv"
pred_df.to_csv(out)
tail_df.to_csv(Path(cfg.output_dir) / "pred_tail_step5.csv")
print(f"Saved pred_df: {out}")

## Phase 3: Step 6 — Daily Retraining (Computational Upper Bound)

Every simulation iteration triggers a **full retrain** of both models:
- XGBoost: cold restart on rolling 250-day window
- OS-ELM: incremental `partial_fit` on the single new row

This is the "always-fresh" baseline — the **upper computational bound**.
If drift-driven retraining beats this on accuracy, the cost savings matter.

In [ ]:
from xgboost import XGBRegressor
import time

X = pd.read_csv(Path(cfg.output_dir) / "X_features.csv", index_col=0)
y = pd.read_csv(Path(cfg.output_dir) / "y_target.csv", index_col=0).squeeze("columns")
pred_df = pd.read_csv(Path(cfg.output_dir) / "predictions_step5.csv", index_col=0)

pred_df["Pred_XGB_Daily"] = 0.0
pred_df["Pred_OSELM_Daily"] = 0.0

xgb = XGBRegressor(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, random_state=42,
)
xgb.fit(X.iloc[:cfg.simulation_start], y.iloc[:cfg.simulation_start])

oselm = OSELM(n_inputs=9, n_hidden=cfg.oselm_hidden, activation="sigmoid")
oselm.fit(X.iloc[:cfg.simulation_start].values, y.iloc[:cfg.simulation_start].values)

print(f"X: {X.shape}, y: {y.shape}, pred_df: {pred_df.shape}")
print("Starting daily retraining loop...")

In [ ]:
start_time = time.perf_counter()
retrain_count = 0

for t in range(cfg.simulation_start, len(X)):
    x_t = X.iloc[t : t + 1]
    y_t = y.iloc[t]

    # Phase 1: Test
    pred_df.iloc[t, 9] = float(xgb.predict(x_t)[0])
    pred_df.iloc[t, 10] = float(oselm.predict(x_t.values)[0])

    # Phase 2: Retrain XGBoost (cold restart, rolling 250-day window)
    lo = max(0, t + 1 - cfg.xgb_window)
    xgb = XGBRegressor(
        n_estimators=100, max_depth=3, learning_rate=0.05,
        subsample=0.8, random_state=42,
    )
    xgb.fit(X.iloc[lo : t + 1], y.iloc[lo : t + 1])

    # Phase 3: Retrain OS-ELM (incremental, 1 new row)
    oselm.partial_fit(x_t.values, np.array([y_t]))

    retrain_count += 1

elapsed = time.perf_counter() - start_time
print(f"Total compute time: {elapsed:.2f}s")

In [ ]:
print("=" * 60)
print("PHASE 3 STEP 6 — DAILY RETRAINING (COMPUTATIONAL UPPER BOUND)")
print("=" * 60)
print()
print(f"Total compute time: {elapsed:.2f}s")
print(f"Total retraining events: {retrain_count}")
print(f"Expected (simulation iterations): {len(X) - cfg.simulation_start}")
print()

sim = pred_df.iloc[cfg.simulation_start:]
xgb_std = sim["Pred_XGB_Daily"].std()
osl_std = sim["Pred_OSELM_Daily"].std()
print(f"Pred_XGB_Daily std (simulation zone): {xgb_std:.6f}")
print(f"Pred_OSELM_Daily std (simulation zone): {osl_std:.6f}")
print()

print("Tail of pred_df (Daily columns) — row by row:")
tail_df = pred_df[["y_true", "Pred_XGB_Daily", "Pred_OSELM_Daily"]].tail()
for idx, row in tail_df.iterrows():
    print(f"  {idx}  {row['y_true']:+.6f}  {row['Pred_XGB_Daily']:+.6f}  {row['Pred_OSELM_Daily']:+.6f}")
print()

out = Path(cfg.output_dir) / "predictions_step6.csv"
pred_df.to_csv(out)
tail_df.to_csv(Path(cfg.output_dir) / "pred_tail_step6.csv")
print(f"Saved pred_df: {out}")

## Phase 4: Comprehensive Metric Evaluation & Computational ROI Analysis
### Step 1 – Protected Custom Accuracy Metrics (RMSE & ε-MAPE)

Computes RMSE and epsilon-protected MAPE across all 10 prediction columns
(5 scenarios × 2 models) on the simulation zone (rows 241–3928).

Also computes standard deviation σ per model to detect:
- OS-ELM plasticity death (frozen predictions → σ ≈ 0)
- Relative prediction variance across retraining strategies

**Formulas:**
- ε-MAPE = mean(|(y₁ᵣᵖₑ₁ - yᵖˢₑ) / (|y₁ᵣᵖₑ₁| + 10⁻⁸)|) × 100
- RMSE   = sqrt(mean((y₁ᵣᵖₑ₁ - yᵖˢₑ)²))

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

output_dir = "../dataset/processed"


def epsilon_mape(y_true, y_pred, eps=1e-8):
    """Epsilon-protected MAPE to avoid ZeroDivisionError."""
    return float(np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + eps))) * 100)


def rmse(y_true, y_pred):
    """Standard Root Mean Squared Error."""
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mae(y_true, y_pred):
    """Mean Absolute Error."""
    return float(np.mean(np.abs(y_true - y_pred)))

In [ ]:
pred_df = pd.read_csv(Path(output_dir) / "predictions_step6.csv", index_col=0, parse_dates=True)
print(f"Loaded: {pred_df.shape[0]} rows \u00d7 {pred_df.shape[1]} columns")

SIM_START = 241
sim = pred_df.iloc[SIM_START:]
print(f"Simulation zone (rows {SIM_START}–{len(pred_df) - 1}): {len(sim)} rows")

pred_cols = sorted([c for c in pred_df.columns if c.startswith("Pred_")])
n_pred = len(pred_cols)
print(f"Prediction columns: {n_pred}")
print(f"Expected: 10 (5 scenarios \u00d7 2 models) \u2192 {'OK' if n_pred == 10 else 'MISMATCH'}")

y_true = sim["y_true"].values

In [ ]:
rows = []
for col in pred_cols:
    y_pred = sim[col].values
    rows.append({
        "Model": col,
        "RMSE": rmse(y_true, y_pred),
        "MAE": mae(y_true, y_pred),
        "Epsilon_MAPE": epsilon_mape(y_true, y_pred),
        "Std_Pred": float(np.std(y_pred, ddof=1)),
    })

metrics_df = pd.DataFrame(rows)

scenario_map = {
    "Pred_XGB_A": "A \u2013 Wasserstein 60",
    "Pred_OSELM_A": "A \u2013 Wasserstein 60",
    "Pred_XGB_B": "B \u2013 Wasserstein 120",
    "Pred_OSELM_B": "B \u2013 Wasserstein 120",
    "Pred_XGB_C": "C \u2013 ADWIN stream",
    "Pred_OSELM_C": "C \u2013 ADWIN stream",
    "Pred_XGB_Static": "Static (no retrain)",
    "Pred_OSELM_Static": "Static (no retrain)",
    "Pred_XGB_Daily": "Daily retrain",
    "Pred_OSELM_Daily": "Daily retrain",
}
metrics_df["Scenario"] = metrics_df["Model"].map(scenario_map)
metrics_df = metrics_df[["Model", "Scenario", "RMSE", "MAE", "Epsilon_MAPE", "Std_Pred"]]

# Max Single-Day MAPE diagnostic for XGB_A
xgb_a_pred = sim["Pred_XGB_A"].values
daily_mape = np.abs((y_true - xgb_a_pred) / (np.abs(y_true) + 1e-8)) * 100
max_1d_mape_xgb_a = float(np.max(daily_mape))
worst_day_idx = int(np.argmax(daily_mape))
worst_day_date = sim.index[worst_day_idx]

print(f"Metrics table: {metrics_df.shape[0]} rows \u00d7 {metrics_df.shape[1]} cols")
print(f"Expected 10 rows \u2192 {'OK' if metrics_df.shape[0] == 10 else 'MISMATCH'}")

In [ ]:
print("=" * 80)
print(f"PHASE 4 STEP 1 \u2014 ACCURACY METRICS REPORT (simulation zone: {len(sim)} rows)")
print("=" * 80)
print()

print("\u2500 10-model comparison \u2500")
print(metrics_df.round(6).to_string(index=False))
print()

print("\u2500 XGBoost only \u2500")
xgb_df = metrics_df[metrics_df["Model"].str.contains("XGB")].copy()
print(xgb_df.round(6).to_string(index=False))
print()

print("\u2500 OS-ELM only \u2500")
osl_df = metrics_df[metrics_df["Model"].str.contains("OSELM")].copy()
print(osl_df.round(6).to_string(index=False))
print()

# Plasticity death detection
osl_static_std = osl_df.loc[osl_df["Model"] == "Pred_OSELM_Static", "Std_Pred"].values[0]
osl_daily_std = osl_df.loc[osl_df["Model"] == "Pred_OSELM_Daily", "Std_Pred"].values[0]
print("\u2500 Plasticity death check \u2500")
print(f"OS-ELM Static \u03c3: {osl_static_std:.8f}")
print(f"OS-ELM Daily \u03c3:   {osl_daily_std:.8f}")
if osl_static_std < 1e-5:
    print("\u26a0 Plasticity death CONFIRMED \u2014 static OS-ELM predictions frozen at constant value")
else:
    print("\u2713 No plasticity death in static OS-ELM")
print()

print("\u2500 XGB_A worst-day diagnostic \u2500")
print(f"Max Single-Day MAPE for Pred_XGB_A: {max_1d_mape_xgb_a:.0f}%")
print(f"Worst day: {worst_day_date.date()} (simulation row {worst_day_idx})")
print()

# Persist
out_path = Path(output_dir) / "metrics_accuracy_df.csv"
metrics_df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

## Phase 4: Step 2 — Computational ROI Analysis (Accuracy vs Efficiency Trade-off)

Marries accuracy (MAE) from Step 1 with operational runtime from Phase 3
to answer: “How much accuracy must we sacrifice for extreme time savings?”

**Key Metrics:**
- Time_Saved% relative to Daily retrain (computational upper bound)
- MAE_Penalty% relative to same-model-group Daily retrain
  → XGBoost models penalized against Pred_XGB_Daily MAE
  → OS-ELM models penalized against Pred_OSELM_Daily MAE

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

output_dir = "../dataset/processed"

metrics_df = pd.read_csv(Path(output_dir) / "metrics_accuracy_df.csv")

# Runtime per scenario from Phase 3 (shared across both models)
runtime_map = {
    "Pred_XGB_Static": 14.34,    "Pred_OSELM_Static": 14.34,
    "Pred_XGB_A": 127.62,        "Pred_OSELM_A": 127.62,
    "Pred_XGB_B": 95.37,         "Pred_OSELM_B": 95.37,
    "Pred_XGB_C": 19.21,         "Pred_OSELM_C": 19.21,
    "Pred_XGB_Daily": 501.90,    "Pred_OSELM_Daily": 501.90,
}
DAILY_RUNTIME = 501.90

# Per-model-group Daily MAE reference
mae_lookup = metrics_df.set_index("Model")["MAE"].to_dict()
mae_xgb_daily = mae_lookup["Pred_XGB_Daily"]
mae_osl_daily = mae_lookup["Pred_OSELM_Daily"]


def daily_ref(model_name: str) -> float:
    return mae_xgb_daily if "XGB" in model_name else mae_osl_daily


roi_rows = []
for _, row in metrics_df.iterrows():
    model = row["Model"]
    runtime = runtime_map[model]
    time_saved = ((DAILY_RUNTIME - runtime) / DAILY_RUNTIME) * 100
    ref = daily_ref(model)
    mae_penalty = ((row["MAE"] - ref) / ref) * 100
    roi_rows.append({
        "Model": model,
        "Scenario": row["Scenario"],
        "Runtime_s": runtime,
        "Time_Saved_%": round(time_saved, 2),
        "MAE": row["MAE"],
        "MAE_Penalty_%": round(mae_penalty, 2),
    })

roi_df = pd.DataFrame(roi_rows)
roi_df = roi_df[["Model", "Scenario", "Runtime_s", "Time_Saved_%", "MAE", "MAE_Penalty_%"]

print(f"ROI table: {roi_df.shape[0]} rows \u00d7 {roi_df.shape[1]} cols")

roi_df.to_csv(Path(output_dir) / "roi_analysis.csv", index=False)
print("Saved: ../dataset/processed/roi_analysis.csv")

In [ ]:
print("=" * 85)
print("PHASE 4 STEP 2 \u2014 COMPUTATIONAL ROI ANALYSIS")
print("=" * 85)
print()

print("\u2500 Combined (10 models) \u2500")
print(roi_df.round(4).to_string(index=False))
print()

print("\u2500 XGBoost only \u2500")
xgb_roi = roi_df[roi_df["Model"].str.contains("XGB")].copy().reset_index(drop=True)
print(xgb_roi.round(4).to_string(index=False))
print()

print("\u2500 OS-ELM only \u2500")
osl_roi = roi_df[roi_df["Model"].str.contains("OSELM")].copy().reset_index(drop=True)
print(osl_roi.round(4).to_string(index=False))
print()

# Key insights
print("\u2500 Key insights \u2500")
xgb_a_penalty = xgb_roi.loc[xgb_roi["Model"] == "Pred_XGB_A", "MAE_Penalty_%"].values[0]
xgb_a_time = xgb_roi.loc[xgb_roi["Model"] == "Pred_XGB_A", "Time_Saved_%"].values[0]
print(f"Worst penalty:  XGB_A  \u2192 MAE +{xgb_a_penalty:.1f}%  (saves {xgb_a_time:.1f}% time)")

osl_c_penalty = osl_roi.loc[osl_roi["Model"] == "Pred_OSELM_C", "MAE_Penalty_%"].values[0]
osl_c_time = osl_roi.loc[osl_roi["Model"] == "Pred_OSELM_C", "Time_Saved_%"].values[0]
print(f"Best trade-off: OS-ELM_C \u2192 MAE {osl_c_penalty:+.2f}%  (saves {osl_c_time:.1f}% time)")

osl_static_penalty = osl_roi.loc[osl_roi["Model"] == "Pred_OSELM_Static", "MAE_Penalty_%"].values[0]
extra = "(negative = better than Daily!)" if osl_static_penalty < 0 else ""
print(f"OS-ELM Static:    MAE {osl_static_penalty:+.2f}%  {extra}")

## Phase 4: Step 3 — IFTK Journal-Standard Graphic Visualization

Converts accuracy numbers into publication-ready time-series plots
showing how drift-driven retraining (ADWIN / Scenario C) corrects
prediction errors during the COVID-19 market crash (2020).

**Plot protocol (per research plan §VI.A.4):**
- Decoupled figures: one per model group (XGBoost, OS-ELM)
- Each figure: y_true + Static (no retrain) + Daily + C (hero)
- Red dashed vertical lines = ADWIN drift alarm dates
- 300 DPI, print-friendly palette, clear date axis

**Narrative:** If drift-driven correction works, the C line should
visibly kink toward y_true after each red dashed alarm.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

output_dir = "../dataset/processed"

# Load data
pred_df = pd.read_csv(Path(output_dir) / "predictions_step6.csv", index_col=0, parse_dates=True)
drift = pd.read_csv(Path(output_dir) / "drift_signals_aligned.csv", index_col=0, parse_dates=True)

# Filter to 2020
mask_2020 = pred_df.index.year == 2020
pred_2020 = pred_df.loc[mask_2020].copy()
n_2020 = len(pred_2020)
print(f"2020 rows: {n_2020}  ({pred_2020.index[0].date()} to {pred_2020.index[-1].date()})")

# ADWIN alarms in 2020
drift_c = drift.loc[mask_2020, "drift_C"]
alarm_dates = drift_c[drift_c == 1.0].index
n_alarms = len(alarm_dates)
print(f"ADWIN drift alarms in 2020: {n_alarms}")
for d in alarm_dates:
    print(f"  {d.date()}")
print()

# Pre-/post-alarm error diagnostics
y_2020 = pred_2020["y_true"].values
if n_alarms >= 1:
    first = alarm_dates[0]
    idx = pred_2020.index.get_loc(first)
    lo, hi = max(0, idx - 10), min(len(pred_2020), idx + 10)
    print("MAE 10-day window before vs after first alarm:")
    for tag, col in [("XGB_C", "Pred_XGB_C"), ("OSELM_C", "Pred_OSELM_C"),
                      ("XGB_S", "Pred_XGB_Static"), ("OSELM_S", "Pred_OSELM_Static")]:
        bef = np.mean(np.abs(y_2020[lo:idx] - pred_2020[col].values[lo:idx]))
        aft = np.mean(np.abs(y_2020[idx:hi] - pred_2020[col].values[idx:hi]))
        chg = ((aft - bef) / bef * 100) if bef > 0 else 0
        label = "better" if chg < 0 else "worse"
        print(f"  {tag}: before={bef:.6f}  after={aft:.6f}  ({chg:+.1f}%, {label})")
    print()

# Style
plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 300, "font.size": 10,
                      "axes.grid": True, "grid.alpha": 0.3})

# Figure 1: XGBoost
fig1, ax1 = plt.subplots(figsize=(14, 5))
ax1.plot(pred_2020.index, pred_2020["y_true"], color="black", linewidth=0.8, label="y_true (actual)")
ax1.plot(pred_2020.index, pred_2020["Pred_XGB_Static"], color="gray", linewidth=1.0,
         linestyle="--", alpha=0.7, label="XGB Static (no retrain)")
ax1.plot(pred_2020.index, pred_2020["Pred_XGB_Daily"], color="#2166AC", linewidth=1.0,
         label="XGB Daily retrain")
ax1.plot(pred_2020.index, pred_2020["Pred_XGB_C"], color="#B2182B", linewidth=1.5,
         label="XGB C (ADWIN-driven)")
for d in alarm_dates:
    ax1.axvline(x=d, color="red", linestyle="--", alpha=0.5, linewidth=1.0)
ax1.set_title("Figure 1: XGBoost Predictions During COVID-19 Crash (2020)", fontsize=12, fontweight="bold")
ax1.set_ylabel("Log Return")
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha="right")
ax1.legend(fontsize=9, framealpha=0.9)
fig1.tight_layout()
fig1.savefig(Path(output_dir) / "fig1_xgboost_2020.png")
plt.show()
print("Saved: fig1_xgboost_2020.png")

# Figure 2: OS-ELM
fig2, ax2 = plt.subplots(figsize=(14, 5))
ax2.plot(pred_2020.index, pred_2020["y_true"], color="black", linewidth=0.8, label="y_true (actual)")
ax2.plot(pred_2020.index, pred_2020["Pred_OSELM_Static"], color="gray", linewidth=1.0,
         linestyle="--", alpha=0.7, label="OS-ELM Static (no retrain)")
ax2.plot(pred_2020.index, pred_2020["Pred_OSELM_Daily"], color="#2166AC", linewidth=1.0,
         label="OS-ELM Daily retrain")
ax2.plot(pred_2020.index, pred_2020["Pred_OSELM_C"], color="#B2182B", linewidth=1.5,
         label="OS-ELM C (ADWIN-driven)")
for d in alarm_dates:
    ax2.axvline(x=d, color="red", linestyle="--", alpha=0.5, linewidth=1.0)
ax2.set_title("Figure 2: OS-ELM Predictions During COVID-19 Crash (2020)", fontsize=12, fontweight="bold")
ax2.set_ylabel("Log Return")
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha="right")
ax2.legend(fontsize=9, framealpha=0.9)
fig2.tight_layout()
fig2.savefig(Path(output_dir) / "fig2_oselm_2020.png")
print("Saved: fig2_oselm_2020.png")
print()

# Report
print("=" * 60)
print("PHASE 4 STEP 3 -- VISUALIZATION REPORT")
print("=" * 60)
print()
ok1 = Path(output_dir, "fig1_xgboost_2020.png").exists()
ok2 = Path(output_dir, "fig2_oselm_2020.png").exists()
print(f"Execution: {'SUCCESS' if ok1 and ok2 else 'PARTIAL'}")
print(f"  fig1_xgboost_2020.png: {'OK' if ok1 else 'MISSING'}")
print(f"  fig2_oselm_2020.png:   {'OK' if ok2 else 'MISSING'}")
print()
osl_s = pred_2020["Pred_OSELM_Static"]
print(f"OS-ELM Static 2020 range: [{osl_s.min():.6f}, {osl_s.max():.6f}]")
print(f"  Plasticity death: {'VISIBLE (std near zero)' if osl_s.std() < 1e-5 else 'NOT visible'}")
print()
if n_alarms > 0:
    xgb_c_r = float(np.sqrt(np.mean((pred_2020["Pred_XGB_C"] - y_2020) ** 2)))
    xgb_d_r = float(np.sqrt(np.mean((pred_2020["Pred_XGB_Daily"] - y_2020) ** 2)))
    osl_c_r = float(np.sqrt(np.mean((pred_2020["Pred_OSELM_C"] - y_2020) ** 2)))
    osl_d_r = float(np.sqrt(np.mean((pred_2020["Pred_OSELM_Daily"] - y_2020) ** 2)))
    print("RMSE ratios (2020 zone, C vs Daily):")
    print(f"  XGB_C  / XGB_Daily:   {xgb_c_r / xgb_d_r:.3f}x")
    print(f"  OSELM_C / OSELM_Daily: {osl_c_r / osl_d_r:.3f}x")
    print()
    if xgb_c_r < xgb_d_r:
        print("XGB_C outperforms XGB_Daily in 2020 -- ADWIN correction effective")
    elif xgb_c_r < xgb_d_r * 1.2:
        print("XGB_C close to XGB_Daily in 2020 -- ADWIN correction competitive")
    else:
        print("XGB_C lags XGB_Daily in 2020 -- fewer retrains hurt accuracy")
else:
    print("No ADWIN alarms in 2020 -- cannot assess drift-driven correction")